<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# **AttGAN on CelebA** 
**Dataset:** CelebA (torchvision) · **Framework:** PyTorch · **Platform:** Google Colab
> He et al. (2019) — *AttGAN: Facial Attribute Editing by Only Changing What You Want*

- Aissa Berenice González Fosado
- Daniela de la Torre Gallo
- Clara Paola Aguilar Casillas

**Experiments**

| Experiment | λ_rec | λ_cls_G | Purpose |
|---|---|---|---|
| `exp1_baseline` | 100 | 1 | Paper defaults — reference |
| `exp2_high_rec` | 200 | 1 | Stronger identity preservation |
| `exp3_strong_attr` | 50 | 5 | Sharper attribute edits |


## *Clone repo & install*

In [ ]:
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git 2>/dev/null || echo 'Repo already cloned'
%cd GAN_Project_DL
!pip install -q -r requirements.txt
print('Setup complete')


## *Verify GPU*

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change Runtime Type > T4 GPU'
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## *Select experiment*


In [ ]:
import importlib, torch

# ── CHANGE THIS ─────────────────────────────────────────────────────
EXPERIMENT = 'exp1_baseline'
# Options: 'exp1_baseline' | 'exp2_high_rec' | 'exp3_strong_attr'
# ────────────────────────────────────────────────────────────────────

_MAP = {
    'exp1_baseline':    ('experiments.exp1_baseline', 'Exp1Config'),
    'exp2_high_rec':    ('experiments.exp2_high_rec', 'Exp2Config'),
    'exp3_strong_attr': ('experiments.exp3_low_rec',  'Exp3Config'),
}
mod_path, cls_name = _MAP[EXPERIMENT]
cfg = getattr(importlib.import_module(mod_path), cls_name)()
device = torch.device('cuda')

print(f'Experiment   : {cfg.EXPERIMENT_NAME}')
print(f'lambda_rec   : {cfg.LAMBDA_REC}')
print(f'lambda_cls_D : {cfg.LAMBDA_CLS_D}')
print(f'lambda_cls_G : {cfg.LAMBDA_CLS_G}')
print(f'Epochs       : {cfg.N_EPOCHS}')
print(f'Batch size   : {cfg.BATCH_SIZE}')
if hasattr(cfg, 'DESCRIPTION'): print(f'Description  : {cfg.DESCRIPTION}')


## *Load CelebA*
torchvision downloads automatically (~1.4 GB, first run only).


In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}')
print(f'Attr  values: {set(attrs.unique().tolist())}  (should be {{-1.0, 1.0}})')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')


## *Build models*
```
image  ->  Encoder (5x Conv2d stride-2)  ->  z (512x4x4)
                                                  |
                          target_attrs -----------+ (tiled, concat)
                                                  |
                                   Generator (5x ConvTranspose2d) -> fake_image
                                                                           |
                                   Discriminator  <------------------------+
                                     |- adv_head  MSELoss (LSGAN)
                                     |- cls_head  BCEWithLogitsLoss (13 attrs)
```

In [ ]:
from src.models import build_models
enc, gen, dis = build_models(cfg, device)


## *Train*
| Loss | Function | Weight | Purpose |
|---|---|---|---|
| L_adv | MSELoss (LSGAN) | — | Realism |
| L_cls | BCEWithLogitsLoss | λ_cls_D / λ_cls_G | Correct attributes |
| L_rec | L1Loss | λ_rec | Identity preservation |

Samples + checkpoints saved every 5 epochs to `results/<experiment>/`.

In [ ]:
from src.trainer import Trainer

trainer = Trainer(enc, gen, dis, train_loader, test_loader, cfg, device)

# To resume from checkpoint:
# g_losses, d_losses = trainer.train(resume_path='checkpoints/exp1_baseline/ckpt_epoch010.pt')

g_losses, d_losses = trainer.train()

## *Loss curves*

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

## *Attribute manipulation demo*
Each row = one test image. Each column = one attribute toggled independently.  
The model should change **only** the requested attribute.

In [ ]:
from src.utils import attribute_demo
attribute_demo(enc, gen, test_loader, cfg, n_imgs=4)


## *Attribute accuracy & reconstruction L1*

In [ ]:
from src.utils import evaluate_attribute_accuracy, evaluate_reconstruction

acc = evaluate_attribute_accuracy(enc, gen, dis, test_loader, cfg)
rec = evaluate_reconstruction(enc, gen, test_loader, cfg)

print(f'Attribute accuracy : {acc:.1f}%')
print(f'Reconstruction L1  : {rec:.4f}')


## *FID & DACID*
**FID** — Frechet Inception Distance (Heusel et al. 2017). Lower = better.  
**DACID** — custom metric: L2 of mean Inception features. Lower = better.


In [ ]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics
    scores = compute_metrics(enc, gen, test_loader, cfg, device)
    print(f"FID   : {scores['fid']}")
    print(f"DACID : {scores['dacid']}")
else:
    scores = {}
    print('Metrics skipped.')


## *Save metrics.json*
**Run this at the end of every experiment.**  


In [ ]:
import json

payload = {
    'experiment':   cfg.EXPERIMENT_NAME,
    'model':        'AttGAN',
    'lambda_rec':   cfg.LAMBDA_REC,
    'lambda_cls_d': cfg.LAMBDA_CLS_D,
    'lambda_cls_g': cfg.LAMBDA_CLS_G,
    'fid':          scores.get('fid'),
    'dacid':        scores.get('dacid'),
    'attr_acc':     round(float(acc), 2) if 'acc' in dir() else None,
    'rec_l1':       round(float(rec), 4) if 'rec' in dir() else None,
    'g_losses':     g_losses,
    'd_losses':     d_losses,
}
out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Saved -> {out}')
for k in ['lambda_rec','lambda_cls_g','fid','dacid','attr_acc','rec_l1']:
    print(f'  {k:<16}: {payload[k]}')


## *Download results ZIP*

In [ ]:
import shutil
from google.colab import files
zip_name = f'{cfg.EXPERIMENT_NAME}_results'
shutil.make_archive(zip_name, 'zip', cfg.RESULTS_DIR)
files.download(f'{zip_name}.zip')


## *Compare all experiments*
** after all three experiments are done.**

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('export_results', 'export_results.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

## *Fallback* 
If the CelebA quota exceeded

Use one of these if Cell 3 raises `FileURLRetrievalError: Too many users`.


In [ ]:
# 1. Go to kaggle.com > Account > Create New Token  (downloads kaggle.json)
# 2. Upload kaggle.json via Colab Files panel (left sidebar)
# 3. Uncomment and run:

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d jessicali9530/celeba-dataset -p data/
# !unzip -q data/celeba-dataset.zip -d data/
print('Uncomment above lines after uploading kaggle.json')

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.makedirs('data', exist_ok=True)
# !ln -sfn '/content/drive/MyDrive/YOUR_CELEBA_FOLDER' data/celeba
# # Then re-run Cell 4
print('Uncomment above and set YOUR_CELEBA_FOLDER')